# AsyncBridge Usage

`AsyncBridge` helps you call awaitable functions from synchronous code. This notebook covers basic execution, return values, timeout handling, and calls from inside a running event loop.

In [ ]:
import asyncio
import threading

from qubex.core import AsyncBridge

## 1) Run an awaitable from synchronous code

In [ ]:
async def delayed_print(message: str, delay: float) -> None:
    """Sleep and print one message."""
    await asyncio.sleep(delay)
    print(message)


bridge = AsyncBridge()
bridge.run(lambda: delayed_print("hello", delay=2.0))
bridge.close()

## 2) Return a value

In [ ]:
async def delayed_sum(x: int, y: int, delay: float) -> int:
    """Sleep and return sum of two integers."""
    await asyncio.sleep(delay)
    return x + y


with AsyncBridge() as bridge:
    result = bridge.run(lambda: delayed_sum(2, 3, delay=1.0))

print(f"result = {result}")

## 3) Handle timeout

In [ ]:
with AsyncBridge(default_timeout=0.3) as bridge:
    try:
        bridge.run(lambda: delayed_print("this will timeout", delay=1.0))
    except TimeoutError as exc:
        print(f"Caught expected timeout: {exc}")

## 4) Call from inside a running event loop

When `run()` is called inside an active event loop, the awaitable is executed on the bridge's dedicated loop thread.

In [ ]:
async def report_thread() -> str:
    """Return the current thread name after a short delay."""
    await asyncio.sleep(0.1)
    return threading.current_thread().name


async def main() -> None:
    """Run bridge call from within the notebook event loop."""
    with AsyncBridge(thread_name="async-bridge") as bridge:
        caller_thread = threading.current_thread().name
        bridge_thread = bridge.run(lambda: report_thread())
        print(f"caller thread: {caller_thread}")
        print(f"bridge thread: {bridge_thread}")


await main()